## Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import torch

## Data Loading

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'stroke-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/stroke-prediction-dataset


### Inspecting Downloaded Files

In [ ]:
import os
os.listdir("/kaggle/input/stroke-prediction-dataset")

['healthcare-dataset-stroke-data.csv']

### Loading the Dataset

In [ ]:
data = path + "/healthcare-dataset-stroke-data.csv"

df = pd.read_csv(data)
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


### Data Overview - Head, Tail, and Sample Rows

In [ ]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,Female,80.0,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81.0,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35.0,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51.0,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0
5109,44679,Female,44.0,0,0,Yes,Govt_job,Urban,85.28,26.2,Unknown,0


### Dataset Dimensions

In [ ]:
df.sample(5)

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
1502,15219,Female,27.0,0,0,No,Private,Rural,78.05,22.3,never smoked,0
2441,3668,Female,65.0,0,0,Yes,Govt_job,Urban,84.47,52.7,smokes,0
149,58978,Female,70.0,0,1,Yes,Private,Rural,239.07,26.1,never smoked,1
3015,51421,Female,54.0,0,0,Yes,Private,Rural,65.38,25.9,Unknown,0
104,33175,Female,57.0,0,0,Yes,Govt_job,Urban,110.52,28.5,Unknown,1


In [ ]:
df.shape

(5110, 12)

In [ ]:
df.size

61320

## Data Cleaning and Preprocessing

In [ ]:
# Check for the sum of null values in each column
print("Sum of null values per column:")
display(df.isnull().sum())

Sum of null values per column:


,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


### Checking for Duplicate Rows

In [ ]:
# Check for the sum of duplicate rows
print("Number of duplicate rows:")
display(df.duplicated().sum())

Number of duplicate rows:


np.int64(0)

### Handling Categorical Features

In [ ]:
# Check unique values in 'gender' column
print("Unique values in 'gender' column:")
display(df['gender'].unique())

### Addressing 'Other' Gender and Missing BMI

It looks like the 'gender' column has an 'Other' category, which is rare. We should probably drop this row or handle it appropriately. For now, let's remove it and proceed with imputation for `bmi`.

In [ ]:
# Remove rows where gender is 'Other'
df = df[df['gender'] != 'Other']

# Impute missing 'bmi' values with the mean
df['bmi'].fillna(df['bmi'].mean(), inplace=True)

# Verify that there are no more missing values in 'bmi'
print("Sum of null values per column after BMI imputation:")
display(df.isnull().sum())

Sum of null values per column after BMI imputation:


,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,0


### One-Hot Encoding Categorical Features

Now that missing values are handled, let's encode the categorical features using one-hot encoding and drop the 'id' column, as it does not contribute to prediction.

In [ ]:
# Drop the 'id' column
df.drop('id', axis=1, inplace=True)

# Select categorical features for one-hot encoding
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Display the first few rows of the preprocessed DataFrame
print("DataFrame after one-hot encoding:")
display(df.head())

DataFrame after one-hot encoding:


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.60000,1,True,True,False,True,False,False,True,True,False,False
1,61.0,0,0,202.21,28.89456,1,False,True,False,False,True,False,False,False,True,False
2,80.0,0,1,105.92,32.50000,1,True,True,False,True,False,False,False,False,True,False
3,49.0,0,0,171.23,34.40000,1,False,True,False,True,False,False,True,False,False,True
4,79.0,1,0,174.12,24.00000,1,False,True,False,False,True,False,False,False,True,False


### Feature Scaling Numerical Columns

Now, let's apply feature scaling to the numerical columns to standardize them. This helps in ensuring that all features contribute equally to the model training process.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('stroke', axis=1)
y = df['stroke']

# Identify numerical columns for scaling
# These are the original numerical columns that were not one-hot encoded
numerical_cols = ['age', 'avg_glucose_level', 'bmi']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical columns
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Display the first few rows of the DataFrame with scaled features
print("DataFrame after feature scaling:")
display(X.head())

DataFrame after feature scaling:


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,1.051242,0,1,2.706450,1.001034e+00,True,True,False,True,False,False,True,True,False,False
1,0.785889,0,0,2.121652,4.615423e-16,False,True,False,False,True,False,False,False,True,False
2,1.626174,0,1,-0.004867,4.683922e-01,True,True,False,True,False,False,False,False,True,False
3,0.255182,0,0,1.437473,7.152261e-01,False,True,False,True,False,False,True,False,False,True
4,1.581949,1,0,1.501297,-6.358651e-01,False,True,False,False,True,False,False,False,True,False


### Checking Data Types Before Boolean Conversion

In [ ]:
# Display current data types to see if any boolean columns exist
print("Data types of X DataFrame BEFORE conversion:")
display(X.dtypes)

Data types of X DataFrame BEFORE conversion:


,0
age,float64
hypertension,int64
heart_disease,int64
avg_glucose_level,float64
bmi,float64
gender_Male,bool
ever_married_Yes,bool
work_type_Never_worked,bool
work_type_Private,bool
work_type_Self-employed,bool


### Converting Boolean Columns to Integers

In [ ]:
# Identify boolean columns and convert them to int (0 and 1)
boolean_cols = X.select_dtypes(include=['bool']).columns

if not boolean_cols.empty:
    print(f"Converting the following boolean columns to 0 and 1: {list(boolean_cols)}")
    X[boolean_cols] = X[boolean_cols].astype(int)
    print("Conversion complete.")
else:
    print("No boolean columns found in X DataFrame. Skipping conversion.")

Converting the following boolean columns to 0 and 1: ['gender_Male', 'ever_married_Yes', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
Conversion complete.


### Displaying Data Types After Boolean Conversion

In [ ]:
# Display X DataFrame and its data types AFTER potential conversion
print("X DataFrame AFTER potential boolean conversion:")
display(X.head())
print("Data types of X DataFrame AFTER potential boolean conversion:")
display(X.dtypes)

X DataFrame AFTER potential boolean conversion:


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,1.051242,0,1,2.706450,1.001034e+00,1,1,0,1,0,0,1,1,0,0
1,0.785889,0,0,2.121652,4.615423e-16,0,1,0,0,1,0,0,0,1,0
2,1.626174,0,1,-0.004867,4.683922e-01,1,1,0,1,0,0,0,0,1,0
3,0.255182,0,0,1.437473,7.152261e-01,0,1,0,1,0,0,1,0,0,1
4,1.581949,1,0,1.501297,-6.358651e-01,0,1,0,0,1,0,0,0,1,0


Data types of X DataFrame AFTER potential boolean conversion:


,0
age,float64
hypertension,int64
heart_disease,int64
avg_glucose_level,float64
bmi,float64
gender_Male,int64
ever_married_Yes,int64
work_type_Never_worked,int64
work_type_Private,int64
work_type_Self-employed,int64


## Data Splitting and Resampling

## ANN Model Design and Training with PyTorch

In [ ]:
from sklearn.model_selection import train_test_split

# Convert pandas DataFrame/Series to PyTorch tensors
X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1) # Unsqueeze for binary classification output

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: torch.Size([4087, 15])
y_train shape: torch.Size([4087, 1])
X_test shape: torch.Size([1022, 15])
y_test shape: torch.Size([1022, 1])


### Applying SMOTE for Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE

# Convert tensors back to numpy for SMOTE
X_train_np = X_train.numpy()
y_train_np = y_train.numpy().ravel() # .ravel() converts to 1D array for SMOTE

print(f"Original training set shape: X={X_train_np.shape}, y={y_train_np.shape}")
print(f"Original count of positive class (1): {sum(y_train_np == 1)}")
print(f"Original count of negative class (0): {sum(y_train_np == 0)}")

sm = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_np, y_train_np)

# Convert resampled numpy arrays back to PyTorch tensors
X_train_resampled = torch.tensor(X_train_resampled, dtype=torch.float32)
y_train_resampled = torch.tensor(y_train_resampled, dtype=torch.float32).unsqueeze(1)

print(f"Resampled training set shape: X={X_train_resampled.shape}, y={y_train_resampled.shape}")
print(f"Resampled count of positive class (1): {sum(y_train_resampled == 1)}")
print(f"Resampled count of negative class (0): {sum(y_train_resampled == 0)}")

Original training set shape: X=(4087, 15), y=(4087,)
Original count of positive class (1): 187
Original count of negative class (0): 3900
Resampled training set shape: X=torch.Size([7800, 15]), y=torch.Size([7800, 1])
Resampled count of positive class (1): tensor([3900])
Resampled count of negative class (0): tensor([3900])


## Model Training

### Define the ANN Architecture

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define the ANN model
class ANN(nn.Module):
    def __init__(self, input_size, dropout_rate=0.3):
        super(ANN, self).__init__()
        self.layer_1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate) # Added dropout layer
        self.layer_2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(dropout_rate) # Added dropout layer
        self.layer_3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid() # For binary classification output

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        x = self.dropout1(x) # Apply dropout
        x = self.relu(self.layer_2(x))
        x = self.dropout2(x) # Apply dropout
        x = self.sigmoid(self.layer_3(x))
        return x

# Initialize the model
input_size = X_train.shape[1]
model = ANN(input_size)

print(model)

ANN(
  (layer_1): Linear(in_features=15, out_features=128, bias=True)
  (relu): ReLU()
  (dropout1): Dropout(p=0.3, inplace=False)
  (layer_2): Linear(in_features=128, out_features=64, bias=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (layer_3): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## Model Evaluation

### Define Loss Function and Optimizer

In [ ]:
# Define loss function and optimizer
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Train the Model

In [ ]:
num_epochs = 100

for epoch in range(num_epochs):
    # Forward pass using resampled data
    outputs = model(X_train_resampled)
    loss = criterion(outputs, y_train_resampled)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print("Training complete with SMOTE data.")

Epoch [10/100], Loss: 0.6365
Epoch [20/100], Loss: 0.5608
Epoch [30/100], Loss: 0.4979
Epoch [40/100], Loss: 0.4670
Epoch [50/100], Loss: 0.4565
Epoch [60/100], Loss: 0.4441
Epoch [70/100], Loss: 0.4368
Epoch [80/100], Loss: 0.4267
Epoch [90/100], Loss: 0.4187
Epoch [100/100], Loss: 0.4097
Training complete with SMOTE data.


### Evaluate the Model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Set model to evaluation mode
model.eval()

with torch.no_grad(): # Disable gradient calculation for evaluation
    # Predictions on the test set
    test_outputs = model(X_test)
    predicted = (test_outputs > 0.5).float() # Convert probabilities to binary predictions

    # Convert tensors to numpy arrays for sklearn metrics
    y_test_np = y_test.numpy()
    predicted_np = predicted.numpy()

    # Calculate metrics
    accuracy = accuracy_score(y_test_np, predicted_np)
    precision = precision_score(y_test_np, predicted_np)
    recall = recall_score(y_test_np, predicted_np)
    f1 = f1_score(y_test_np, predicted_np)

    print(f'Accuracy of the model on the test set: {accuracy:.4f}')
    print(f'Precision of the model on the test set: {precision:.4f}')
    print(f'Recall of the model on the test set: {recall:.4f}')
    print(f'F1-Score of the model on the test set: {f1:.4f}')

Accuracy of the model on the test set: 0.7515
Precision of the model on the test set: 0.1571
Recall of the model on the test set: 0.7097
F1-Score of the model on the test set: 0.2573


## Notebook Summary

## Addressing Class Imbalance with SMOTE

In [ ]:
# Install imbalanced-learn library if not already installed
!pip install imbalanced-learn

This notebook demonstrates a complete workflow for predicting stroke risk using a tabular dataset. It covers data loading, preprocessing steps like handling missing values and categorical features, feature scaling, addressing class imbalance with SMOTE, defining and training an Artificial Neural Network (ANN) using PyTorch, and finally, evaluating the model's performance.

Now, let's apply SMOTE to our training data to handle the class imbalance. This will generate synthetic samples for the minority class (stroke) to create a more balanced dataset for training.